In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    "bolt://neo4j_bank_lab:7687",
    auth=("neo4j", "test1234")
)

query = """
MATCH (u1:User)-[:USES]->(d:Device)<-[:USES]-(u2:User)
WHERE u1.name < u2.name
  AND EXISTS {
    MATCH (u1)-[t1:TRANSFER]->()
    WHERE t1.timestamp >= datetime() - duration('P60D')
  }
  AND EXISTS {
    MATCH (u2)-[t2:TRANSFER]->()
    WHERE t2.timestamp >= datetime() - duration('P60D')
  }
RETURN u1.name AS user1, u2.name AS user2, d.name AS shared_device
ORDER BY shared_device, user1, user2
"""

with driver.session() as session:
    result = session.run(query)
    rows = [record.data() for record in result]

driver.close()

rows